# GBIF Amphibian Occurrence Records
## Exploratory Data Analysis (EDA)

**Author:** Felipe Andrade

**Project:** GBIF Amphibian ETL Pipeline

**Tools:** Python • Pandas • NumPy • Matplotlib • Plotly

**Related Repository**

This notebook uses the dataset generated in the SQL ETL pipeline:

👉 https://github.com/felipeandrade91/gbif-amphibians-etl-pipeline
---

## Project Overview

This notebook presents an exploratory data analysis (EDA) of amphibian occurrence records obtained from the Global Biodiversity Information Facility (GBIF).

The dataset was previously cleaned and standardized through a PostgreSQL ETL pipeline, where data quality issues, inconsistencies and derived temporal variables were addressed.

The objective of this notebook is to explore the cleaned dataset and identify temporal, taxonomic, geographic and institutional patterns while demonstrating Python-based data analysis and visualization workflows.

### Workflow

```
Raw GBIF Data
        │
        ▼
 PostgreSQL ETL
        │
        ▼
 Clean Dataset
        │
        ▼
Python (EDA + Visualization)
        │
        ▼
Insights
```

## Import Libraries

In [ ]:
## Import Libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import plotly.express as px

from pathlib import Path

# Matplotlib style
plt.style.use("ggplot")

# Pandas display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.precision", 2)

## Load Dataset

In [ ]:
from pathlib import Path

DATA_PATH = Path("../data/gbif_analysis_202606251606.csv")

df = pd.read_csv(DATA_PATH)

## Dataset Overview

In [ ]:
df.head()

## Dataset Structure

In [ ]:
df.info()

## Data Type Standardization

The dataset was exported from PostgreSQL as a CSV file. Therefore, some variables require explicit type conversion before analysis.

This step converts temporal variables to their appropriate data types and ensures consistency throughout the notebook.

In [ ]:
# Convert date column

df["eventdate_clean"] = pd.to_datetime(df["eventdate_clean"])

# Convert temporal variables

integer_columns = [
    "year",
    "month",
    "day",
    "decade",
    "century"
]

for col in integer_columns:
    df[col] = df[col].astype("Int64")

## Dataset Dimensions

The cleaned dataset contains amphibian occurrence records obtained from GBIF after the SQL ETL process.

In [ ]:
print(f"Number of records : {df.shape[0]:,}")
print(f"Number of variables: {df.shape[1]}")

## Data Quality Assessment

This section evaluates the completeness of each variable by summarizing missing values, completeness percentage and data types.

This assessment provides an overview of data quality and helps identify variables that may require additional preprocessing or should be interpreted with caution during subsequent analyses.

In [ ]:
# Data quality summary

data_quality = (
    pd.DataFrame({
        "Variable": df.columns,
        "Data Type": df.dtypes.astype(str),
        "Non-missing": df.notna().sum().values,
        "Missing": df.isna().sum().values,
        "Completeness (%)": (df.notna().mean() * 100).round(2).values,
        "Missing (%)": (df.isna().mean() * 100).round(2).values
    })
    .sort_values("Completeness (%)", ascending=False)
    .reset_index(drop=True)
)

data_quality

In [ ]:
data_quality.style.format({
    "Non-missing": "{:,}",
    "Missing": "{:,}",
    "Completeness (%)": "{:.2f}",
    "Missing (%)": "{:.2f}"
}).background_gradient(
    subset=["Missing (%)"],
    cmap="Reds"
)

### Interpretation

The data quality assessment indicates that taxonomic variables (e.g., `scientificname`, `family`, and `genus`) are nearly complete, reflecting the high taxonomic consistency of the dataset.

In contrast, geographic coordinates (`decimallatitude` and `decimallongitude`) show substantially lower completeness, a common characteristic of historical biodiversity collections where precise georeferencing was often unavailable.

Overall, the dataset exhibits high completeness for the variables required for temporal and taxonomic analyses, while spatial analyses should account for the reduced availability of geographic coordinates.

## Dataset Overview

This section provides a general overview of the amphibian occurrence dataset, including its size and structure.

In [ ]:
df.shape
df.head()
df.info()

## Taxonomic Composition

This section explores the taxonomic structure of the dataset.

In [ ]:
## Family distribution
df["family"].value_counts().head(20)

In [ ]:
## Genus distribution
df["genus"].value_counts().head(20)


In [ ]:
## Species distribution
df["species"].value_counts().head(20)

## Temporal Distribution of Records

This section evaluates how occurrence records are distributed over time.

In [ ]:
df["year"].value_counts().sort_index().plot(kind="line")

## Seasonal Patterns of Occurrence

This section investigates seasonal patterns in amphibian occurrence records.

In [ ]:
df["season"].value_counts()

## Spatial Coverage

This section evaluates the geographic distribution of records.

In [ ]:
df.plot(
    kind="scatter",
    x="decimallongitude",
    y="decimallatitude",
    s=1,
    alpha=0.2
)

## Spatial Outliers in Geographic Coordinates

During the exploratory spatial visualization, most occurrence records are concentrated within the expected geographic extent of Brazil, forming a coherent spatial pattern consistent with amphibian distributions in the Neotropical region.

However, a small number of records appear as isolated outliers outside the main geographic range, including locations far from South America.

These outliers likely reflect:
- Errors in coordinate entry (e.g., swapped latitude/longitude values)
- Default or placeholder coordinates used during data digitization
- Incomplete or low-quality georeferencing from historical records
- Potential issues inherited from original data publishers

Given their low frequency and high likelihood of being artefacts, these records should be treated with caution in spatial analyses and may require filtering or validation in subsequent steps of the pipeline.

In [ ]:
## outliers
df[
    (df["decimallatitude"].abs() > 35) |
    (df["decimallongitude"].abs() > 75)
][["gbifid", "decimallatitude", "decimallongitude"]].head(20)

In [ ]:
df["geo_outlier"] = ~(
    df["decimallatitude"].between(-35, 10) &
    df["decimallongitude"].between(-75, -30)
)

df["geo_outlier"].value_counts()

## Spatial Data Cleaning Strategy

Based on the exploratory spatial analysis, a subset of occurrence records presented geographic coordinates outside the expected spatial extent of Brazil and the Neotropical region.

These records were classified as spatial outliers and considered likely to represent data quality issues rather than true biological occurrences.

### Criteria for spatial filtering

To improve the reliability of subsequent spatial analyses, the following strategy was adopted:

- Retain records within a biologically plausible geographic window for Brazilian amphibians:
  - Latitude: -35 to 10
  - Longitude: -75 to -30

- Flag all records outside this range as potential spatial outliers
- Avoid immediate deletion to preserve traceability and allow future sensitivity analyses

### Rationale

This approach balances data cleaning with data preservation, ensuring that:
- High-confidence records remain available for ecological inference
- Potentially informative but uncertain records are not irreversibly discarded
- Downstream analyses (e.g., richness maps, distribution modeling) are not biased by clear georeferencing errors

This step is particularly important in biodiversity datasets derived from aggregated sources such as GBIF, where spatial accuracy varies significantly across institutions and collection periods.

In [ ]:
## Quality flag
df["spatial_quality"] = "valid"

df.loc[
    (df["decimallatitude"] < -35) |
    (df["decimallatitude"] > 10) |
    (df["decimallongitude"] < -75) |
    (df["decimallongitude"] > -30),
    "spatial_quality"
] = "outlier"

In [ ]:
## Distribution check
df["spatial_quality"].value_counts()

## Spatial Filtering Sensitivity Analysis

To evaluate the impact of spatial data filtering decisions on downstream ecological inference, a sensitivity analysis was performed comparing the full dataset (including spatial outliers) with the filtered dataset (restricted to biologically plausible geographic bounds).

This step is essential in biodiversity data workflows, as spatial filtering can influence:
- estimates of species richness
- spatial distribution patterns
- sampling bias interpretation
- ecological niche inference

The goal is to assess whether excluding spatial outliers materially changes the structure and interpretation of the dataset.

In [ ]:
## Comparison of the number of records
total_records = len(df)

valid_records = df["spatial_quality"].value_counts().get("valid", 0)
outliers = df["spatial_quality"].value_counts().get("outlier", 0)

summary = pd.DataFrame({
    "Category": ["Total", "Valid", "Outliers"],
    "Records": [total_records, valid_records, outliers]
})

summary

In [ ]:
##Visualization

import matplotlib.pyplot as plt

summary.set_index("Category")["Records"].plot(
    kind="bar"
)

plt.title("Impact of Spatial Filtering on Dataset Size")
plt.ylabel("Number of Records")
plt.show()

### Geographic distribution: before and after

In [ ]:
## Before (full dataset)
df.plot(
    kind="scatter",
    x="decimallongitude",
    y="decimallatitude",
    s=1,
    alpha=0.15
)
plt.title("Full Dataset (Including Spatial Outliers)")
plt.show()

In [ ]:
## After (filtered dataset)
df[df["spatial_quality"] == "valid"].plot(
    kind="scatter",
    x="decimallongitude",
    y="decimallatitude",
    s=1,
    alpha=0.15
)
plt.title("Filtered Dataset (Valid Records Only)")
plt.show()

### Wealth effect by state

In [ ]:
full = df["stateprovince_clean"].value_counts()

filtered = df[df["spatial_quality"] == "valid"]["stateprovince_clean"].value_counts()

comparison = pd.DataFrame({
    "Full Dataset": full,
    "Filtered Dataset": filtered
}).fillna(0)

comparison

## Interpretation of Spatial Filtering Effects

The spatial filtering procedure removed a small proportion of records classified as geographic outliers.

Despite their low frequency, these records had the potential to introduce noise in spatial analyses and distort ecological patterns.

The comparison between the full and filtered datasets indicates that:
- The overall spatial structure remains stable after filtering
- No major shifts in regional dominance patterns were observed
- The filtering step improves data reliability without significantly reducing sample size

This suggests that the applied spatial filtering strategy is conservative and appropriate for downstream ecological analyses.

# Sampling Bias Analysis

Biodiversity datasets are rarely collected through probabilistic sampling designs. Instead, occurrence records often reflect historical collecting efforts, institutional priorities, accessibility, and recent contributions from citizen science.

Understanding these sampling biases is essential before drawing ecological conclusions, as observed patterns may represent sampling effort rather than biological processes.

The following analyses explore potential sources of bias in the GBIF amphibian occurrence dataset.

## Temporal Sampling Bias

Occurrence records are expected to increase through time due to the expansion of biological collections, digitization initiatives, and the growth of biodiversity information networks such as GBIF.

Therefore, temporal trends may reflect changes in sampling effort rather than changes in amphibian diversity.

In [ ]:
records_decade = (
    df.groupby("decade")
      .size()
      .reset_index(name="records")
      .dropna()
)

records_decade

In [ ]:
## Temporal Trend of Occurrence Records

plt.figure(figsize=(12,5))

plt.plot(
    records_decade["decade"],
    records_decade["records"],
    marker="o",
    linewidth=2
)

# destaque da década de 1980
plt.axvspan(1970, 1980, alpha=0.20)

# destaque da década de 2020
plt.axvspan(2019, 2029, alpha=0.20)

plt.annotate(
    "Temporary decline",
    xy=(1980,22493),
    xytext=(1962,50000),
    arrowprops=dict(arrowstyle="->")
)

plt.annotate(
    "Incomplete decade",
    xy=(2030,42402),
    xytext=(2000,32000),
    arrowprops=dict(arrowstyle="->")
)

plt.title("Temporal Distribution of GBIF Amphibian Records")

plt.xlabel("Decade")

plt.ylabel("Number of Records")

plt.grid(alpha=.3)

plt.show()

## Institutional Contribution Through Time

Overall temporal trends may conceal changes in the institutions contributing data to GBIF.

A decline in total records during a given decade does not necessarily indicate reduced sampling effort. Instead, it may result from differences in the temporal coverage, digitization status, or publication practices of individual institutions.

To investigate this possibility, the following analyses examine how institutional contributions vary across decades.

In [ ]:
institution_decade = (
    df.groupby(["decade","institutioncode"])
      .size()
      .reset_index(name="records")
)

institution_decade.head()

In [ ]:
top_inst = (
    institution_decade
    .sort_values(["decade","records"], ascending=[True,False])
    .groupby("decade")
    .head(10)
)

top_inst.head(20)

## Institutional Contributions Across Time

To investigate whether temporal fluctuations are associated with changes in data providers, the number of occurrence records was summarized by institution and decade.

A heatmap was used to visualize institutional contributions through time, allowing the identification of periods dominated by specific collections as well as the emergence of new contributors.

If the decline observed during the 1980s is concentrated within a small number of institutions, it would suggest that the pattern reflects changes in data availability or digitization rather than a widespread reduction in amphibian sampling effort.

In [ ]:
## Records by institution and decade

institution_decade = (
    df
    .dropna(subset=["institutioncode", "decade"])
    .groupby(["institutioncode", "decade"])
    .size()
    .reset_index(name="records")
)

institution_decade.head()

In [ ]:
## Top institutions

top20 = (
    df["institutioncode"]
    .value_counts()
    .head(20)
    .index
)

top20

In [ ]:
## Heatmap matrix

heat = (
    institution_decade[
        institution_decade["institutioncode"].isin(top20)
    ]
    .pivot(
        index="institutioncode",
        columns="decade",
        values="records"
    )
    .fillna(0)
)

heat

In [ ]:
## Top institution by decade

leaders = (
    institution_decade
    .sort_values(["decade", "records"], ascending=[True, False])
    .groupby("decade")
    .first()
)

leaders

In [ ]:
## Normalize by institution

heat_pct = heat.div(
    heat.sum(axis=1),
    axis=0
) * 100

## Evolution of Major Data Providers

Changes in the total number of occurrence records through time may reflect shifts in the institutions contributing data to GBIF.

To investigate this pattern, the temporal trajectories of the largest data providers were examined across decades. This analysis helps distinguish changes in biological sampling effort from changes in institutional participation and dataset publication.

In [ ]:
## Top institutions through time

top5 = (
    df["institutioncode"]
    .value_counts()
    .head(5)
    .index
)

top5

In [ ]:
institution_time = (
    df[df["institutioncode"].isin(top5)]
    .groupby(["decade", "institutioncode"])
    .size()
    .reset_index(name="records")
)

institution_time.head()

In [ ]:
plt.figure(figsize=(13,6))

for inst in top5:

    subset = institution_time[
        institution_time["institutioncode"] == inst
    ]

    plt.plot(
        subset["decade"],
        subset["records"],
        marker="o",
        linewidth=2,
        label=inst
    )

plt.title("Temporal Evolution of Major Data Providers")

plt.xlabel("Decade")

plt.ylabel("Number of Occurrence Records")

plt.legend(title="Institution")

plt.grid(alpha=.25)

plt.yscale("log")

plt.tight_layout()

plt.show()

## Evolution of Leading Data Providers

Rather than focusing only on the largest institutions overall, this analysis highlights the institutions that dominated data contributions during different historical periods.

Tracking the leading providers across decades reveals how biodiversity data production evolved over nearly two centuries, reflecting the emergence of new biological collections, digitization initiatives, and, more recently, citizen science platforms.

This perspective provides additional context for interpreting temporal fluctuations in occurrence records observed in the previous section.

In [ ]:
## Institutions that led at least one decade

leaders_list = leaders["institutioncode"].unique()

leaders_list

In [ ]:
institution_history = (
    df[
        df["institutioncode"].isin(leaders_list)
    ]
    .groupby(["decade","institutioncode"])
    .size()
    .reset_index(name="records")
)

institution_history.head()

In [ ]:
plt.figure(figsize=(14,7))

for inst in leaders_list:

    subset = institution_history[
        institution_history["institutioncode"] == inst
    ]

    plt.plot(
        subset["decade"],
        subset["records"],
        marker="o",
        linewidth=2,
        label=inst
    )

plt.title("Evolution of Leading Biodiversity Data Providers")

plt.xlabel("Decade")

plt.ylabel("Number of Records")

plt.grid(alpha=0.30)

plt.legend(
    title="Institution",
    bbox_to_anchor=(1.02,1),
    loc="upper left"
)

plt.tight_layout()

plt.show()

### Interpretation

Although the temporal trend suggests a decline in occurrence records during the 1980s, the institutional analysis indicates that this pattern is more likely associated with changes in data contributors than with a generalized reduction in amphibian sampling effort.

The leading data provider changes substantially through time, reflecting the historical development of biological collections, digitization initiatives, and the publication of institutional datasets in GBIF.

Early decades are dominated by major European and North American museums, whereas Brazilian institutions become the primary contributors after the mid-twentieth century. More recently, citizen-science platforms such as iNaturalist have emerged as the largest single data provider, illustrating the growing importance of community-based biodiversity monitoring.

Overall, these results highlight that temporal patterns observed in aggregated occurrence data should be interpreted in the context of evolving institutional participation rather than solely as changes in biological sampling effort.

# Spatial Sampling Bias

Occurrence records are rarely distributed uniformly across geographic space. Instead, sampling effort is typically concentrated around research institutions, protected areas, accessible regions, and historically well-surveyed locations.

Understanding this spatial bias is essential before drawing ecological conclusions, as observed patterns may reflect differences in sampling effort rather than true biological distributions.

The following analyses explore the geographic distribution and density of amphibian occurrence records available through GBIF.

## Working Dataset

After the spatial quality assessment, all subsequent analyses are performed using only records classified as **valid**. This ensures that geographic summaries, statistical analyses, and visualizations are not influenced by obvious coordinate errors or implausible occurrence locations.

In [ ]:
## Filtered dataset used in subsequent analyses

df_valid = df[df["spatial_quality"] == "valid"].copy()

print(f"Valid records: {len(df_valid):,}")

In [ ]:
## Spatial density of valid occurrence records

plt.figure(figsize=(8,9))

plt.hexbin(
    df_valid["decimallongitude"],
    df_valid["decimallatitude"],
    gridsize=70,
    mincnt=1
)

plt.colorbar(label="Number of records")

plt.xlabel("Longitude")

plt.ylabel("Latitude")

plt.title("Spatial Density of Valid GBIF Amphibian Records")

plt.tight_layout()

plt.show()

In [ ]:
## Number of records by Brazilian state

records_state = (
    df_valid
    .groupby("stateprovince_clean")
    .size()
    .sort_values(ascending=False)
)

records_state

In [ ]:
plt.figure(figsize=(10,8))

records_state.head(20).sort_values().plot.barh()

plt.xlabel("Number of occurrence records")

plt.ylabel("State")

plt.title("Top 20 Brazilian States by Number of Valid Records")

plt.tight_layout()

plt.show()

In [ ]:
top5_share = (
    records_state.head(5).sum()
    / records_state.sum()
    * 100
)

print(f"Top five states account for {top5_share:.1f}% of all valid occurrence records.")

In [ ]:
## Species richness by state

richness_state = (
    df_valid
    .groupby("stateprovince_clean")["species"]
    .nunique()
    .sort_values(ascending=False)
)

richness_state

In [ ]:
plt.figure(figsize=(10,8))

richness_state.head(20).sort_values().plot.barh()

plt.xlabel("Observed species richness")

plt.ylabel("State")

plt.title("Top 20 Brazilian States by Observed Amphibian Richness")

plt.tight_layout()

plt.show()

## Relationship Between Sampling Effort and Observed Richness

A positive relationship is expected between the number of occurrence records and the observed species richness, since increased sampling effort generally leads to the detection of more species.

To quantify this relationship, a simple linear regression was fitted using the number of occurrence records as a proxy for sampling effort and the observed species richness for each Brazilian state.

Although richness is expected to increase with sampling effort, deviations from the fitted relationship may indicate regions that are relatively over- or under-sampled.

In [ ]:
## Linear relationship between sampling effort and observed richness

from scipy.stats import linregress

sampling_vs_richness = (
    pd.DataFrame({
        "records": records_state,
        "richness": richness_state
    })
    .dropna()
)

model = linregress(
    sampling_vs_richness["records"],
    sampling_vs_richness["richness"]
)

r2 = model.rvalue**2

print(f"R² = {r2:.3f}")
print(f"Slope = {model.slope:.4f}")
print(f"P-value = {model.pvalue:.5f}")

In [ ]:
plt.figure(figsize=(9,7))

x = sampling_vs_richness["records"]
y = sampling_vs_richness["richness"]

plt.scatter(
    x,
    y,
    s=60,
    alpha=0.8
)

# linha de regressão
plt.plot(
    x,
    model.intercept + model.slope*x,
    linewidth=2
)

# rotular TODOS os estados
for state in sampling_vs_richness.index:

    plt.text(
        sampling_vs_richness.loc[state, "records"] + 250,
        sampling_vs_richness.loc[state, "richness"],
        state,
        fontsize=8
    )

plt.xlabel("Number of occurrence records")

plt.ylabel("Observed species richness")

plt.title("Sampling Effort versus Observed Species Richness")

# escrever R² no gráfico
plt.text(
    0.05,
    0.95,
    f"$R^2$ = {r2:.3f}\np < {model.pvalue:.001g}",
    transform=plt.gca().transAxes,
    fontsize=11,
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
)

plt.grid(alpha=0.25)

plt.tight_layout()

plt.show()

### Interpretation

The regression analysis revealed a strong positive association between sampling effort and observed species richness across Brazilian states (**R² = 0.73, p < 0.001**), confirming that the number of occurrence records is an important determinant of recorded biodiversity.

Nevertheless, several states deviate substantially from the expected relationship. **Pará, Amazonas, Minas Gerais, Rio de Janeiro, Bahia, Mato Grosso, Espírito Santo, Acre, Goiás, and Santa Catarina** exhibit higher observed species richness than predicted based on sampling effort alone. In contrast, **São Paulo** and **Mato Grosso do Sul** fall below the fitted relationship, indicating fewer recorded species than expected given their large numbers of occurrence records.

These deviations suggest that factors beyond sampling effort—including regional biodiversity, environmental heterogeneity, historical collecting priorities, and taxonomic coverage—also contribute to the observed richness patterns. Consequently, occurrence records should be interpreted as the combined outcome of biological processes and sampling history rather than as direct estimates of biodiversity.

## Residual Analysis

Residuals represent the difference between the observed species richness and the richness predicted by the linear regression model.

Positive residuals indicate states with higher observed richness than expected based on sampling effort, whereas negative residuals indicate lower richness than predicted.

Examining residuals helps identify regions whose biodiversity patterns cannot be explained solely by differences in sampling intensity.

In [ ]:
## Calculate residuals

sampling_vs_richness["predicted"] = (
    model.intercept +
    model.slope * sampling_vs_richness["records"]
)

sampling_vs_richness["residual"] = (
    sampling_vs_richness["richness"] -
    sampling_vs_richness["predicted"]
)

residuals = (
    sampling_vs_richness
    .sort_values("residual")
)

residuals

In [ ]:
plt.figure(figsize=(10,9))

colors = [
    "tab:green" if r > 0 else "tab:red"
    for r in residuals["residual"]
]

plt.barh(
    residuals.index,
    residuals["residual"],
    color=colors
)

plt.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1.2
)

plt.xlabel("Residual (Observed − Predicted Richness)")

plt.title("Residual Species Richness by Brazilian State")

plt.grid(axis="x", alpha=0.25)

plt.tight_layout()

plt.show()

### Interpretation

### Interpretation

Residual analysis provides additional insight into geographic variation after accounting for sampling effort.

Most Brazilian states follow the expected positive relationship between sampling effort and observed species richness. However, several states deviate substantially from the regression model.

**Minas Gerais (+132.9), Bahia (+84.2), Pará (+57.8), Acre (+48.4), Mato Grosso (+43.9), and Rio de Janeiro (+43.6)** exhibit considerably higher species richness than predicted based on sampling effort alone. These positive residuals suggest that factors beyond sampling intensity—such as environmental heterogeneity, regional biodiversity, or historical collecting patterns—contribute to their elevated richness.

Conversely, **São Paulo (-109.2)** stands out as the largest negative residual despite being the most intensively sampled state. This pattern indicates that exceptionally high sampling effort does not necessarily translate into proportionally higher observed richness. Other states, including **Mato Grosso do Sul (-62.5)** and **Rio Grande do Norte (-64.2)**, also present lower richness than expected relative to their sampling effort.

The pronounced negative residual observed for São Paulo is consistent with the extensive historical sampling effort concentrated in the state, where additional collecting is likely to increase the number of occurrence records more rapidly than the number of recorded species.

Overall, these results reinforce that sampling effort explains a substantial proportion of the observed variation in species richness (**R² = 0.73**), but important ecological and historical factors continue to shape biodiversity patterns across Brazilian states.

# Taxonomic Bias

Biodiversity databases are often taxonomically unbalanced, with some taxonomic groups being represented by substantially more records than others.

These differences may arise from unequal species richness, geographic distributions, detectability, research interest, or historical collecting practices.

The following analyses evaluate how occurrence records are distributed among amphibian families, genera, and species, providing an overview of taxonomic representation within the GBIF dataset.

In [ ]:
## Records by Family
family_counts = (
    df_valid["family"]
    .value_counts()
)

family_counts

In [ ]:
plt.figure(figsize=(10,6))

family_counts.head(15).sort_values().plot.barh()

plt.xlabel("Occurrence records")

plt.ylabel("Family")

plt.title("Top Amphibian Families by Number of Records")

plt.tight_layout()

plt.show()

In [ ]:
## Records by genus
genus_counts = (
    df_valid["genus"]
    .value_counts()
)

genus_counts.head(30)

In [ ]:
plt.figure(figsize=(10,7))

genus_counts.head(20).sort_values().plot.barh()

plt.xlabel("Occurrence records")

plt.ylabel("Genus")

plt.title("Top Amphibian Genera by Number of Records")

plt.tight_layout()

plt.show()

In [ ]:
## Species abundance distribution
species_counts = (
    df_valid["species"]
    .value_counts()
)

In [ ]:
plt.figure(figsize=(8,6))

plt.hist(
    species_counts,
    bins=50
)

plt.xlabel("Number of records per species")

plt.ylabel("Number of species")

plt.title("Distribution of Records per Species")

plt.tight_layout()

plt.show()

In [ ]:
## Pareto Analysis
species_counts = species_counts.sort_values(ascending=False)

pareto = (
    species_counts
    .cumsum()
    /
    species_counts.sum()
)

pareto.head()

In [ ]:
n80 = (pareto <= 0.80).sum()

print(f"{n80} species account for 80% of all occurrence records.")

## Species Rank-Abundance and Pareto Analysis

Species were ranked from the most to the least frequently recorded in the dataset.

The rank-abundance curve summarizes taxonomic dominance, while the cumulative (Pareto) curve quantifies how occurrence records are concentrated among species. Together, these visualizations reveal whether biodiversity records are evenly distributed or dominated by a relatively small subset of taxa.

In [ ]:
## Species rank-abundance + Pareto curve

species_counts = (
    df_valid["species"]
    .dropna()
    .value_counts()
    .sort_values(ascending=False)
)

rank = np.arange(1, len(species_counts) + 1)

cumulative = species_counts.cumsum() / species_counts.sum() * 100

rank80 = np.argmax(cumulative >= 80) + 1

In [ ]:
fig, ax1 = plt.subplots(figsize=(11,6))

# Rank-abundance
ax1.plot(
    rank,
    species_counts.values,
    linewidth=2,
    label="Occurrence records"
)

ax1.set_yscale("log")

ax1.set_xlabel("Species rank")

ax1.set_ylabel("Occurrence records (log scale)")

# Pareto
ax2 = ax1.twinx()

ax2.plot(
    rank,
    cumulative,
    linewidth=2
)

ax2.set_ylabel("Cumulative percentage of records")

ax2.set_ylim(0,100)

# Linha dos 80%

ax2.axhline(
    80,
    linestyle="--",
    linewidth=1
)

ax2.axvline(
    rank80,
    linestyle="--",
    linewidth=1
)

ax2.scatter(
    rank80,
    80,
    s=70,
    zorder=5
)

ax2.text(
    rank80+15,
    81,
    f"{rank80} species\n(80% of records)",
    fontsize=9
)

ax1.plot(
    rank,
    species_counts.values,
    color="tab:blue",
    linewidth=2,
    label="Occurrence records"
)

ax2.plot(
    rank,
    cumulative,
    color="tab:red",
    linewidth=2,
    label="Cumulative %"
)

plt.title("Species Rank-Abundance and Pareto Distribution")

plt.tight_layout()

plt.show()

## Interpretation of Taxonomic Bias

The taxonomic structure of the dataset reveals a strong and consistent pattern of uneven representation across amphibian groups.

At the family level, a small number of families—particularly Hylidae, Leptodactylidae, and Bufonidae—dominate the dataset, collectively accounting for a large proportion of all occurrence records. This pattern reflects both the high diversity of these groups in the Neotropics and their historical prominence in biological surveys.

A similar pattern is observed at the genus level, where genera such as *Rhinella*, *Leptodactylus*, *Boana*, and *Scinax* concentrate a substantial fraction of records. This indicates that sampling effort is not evenly distributed across amphibian diversity but is instead concentrated in a subset of well-studied taxa.

Species-level analysis further highlights this imbalance. The rank-abundance distribution shows a steep decline, indicating that a small number of species dominate the dataset. Quantitatively, only **199 species (approximately 12% of all recorded species)** account for **80% of all occurrence records**, demonstrating a strong dominance structure.

Together, these results confirm that the GBIF amphibian dataset is highly skewed toward a relatively small subset of taxa. This taxonomic bias should be considered when using the data for ecological or macroecological analyses, as it may influence estimates of diversity, abundance patterns, and comparative analyses across groups.

## Data Limitations and Analytical Implications

While the GBIF amphibian dataset provides a large-scale and valuable source of biodiversity information, several limitations must be considered when interpreting the results.

### Spatial bias
Occurrence records are highly uneven across geographic space, with strong concentration in well-sampled regions. This introduces spatial sampling bias, which may inflate perceived richness in intensively studied areas.

### Temporal bias
The dataset shows strong temporal heterogeneity, particularly after the mid-20th century, reflecting the digitization of museum collections and the growth of open biodiversity platforms rather than continuous biological sampling.

### Taxonomic and institutional bias
A small subset of taxa and institutions dominates the dataset. This reflects historical research priorities and institutional capacity, rather than true biological abundance or diversity.

### Analytical implications
These biases collectively indicate that GBIF occurrence data should not be interpreted as a direct proxy for biodiversity patterns. Instead, they represent a combination of biological signal and heterogeneous sampling processes, which must be explicitly accounted for in downstream analyses.

## Species Accumulation Curve

Species accumulation curves are widely used in ecology to evaluate whether sampling effort is sufficient to capture the diversity of a region or dataset.

In this analysis, we assess how species richness increases as additional occurrence records are incorporated. A curve that approaches an asymptote suggests sampling saturation, whereas a continuously increasing curve indicates that additional sampling would likely reveal new species.

In [ ]:
## Species Accumulation Curve - data preparation
## Goal: evaluate how species richness increases with sampling effort

import numpy as np
import matplotlib.pyplot as plt

df_valid = df[df["spatial_quality"] == "valid"].copy()

# sort dataset to simulate incremental sampling process
df_sorted = df_valid.sample(frac=1, random_state=42).reset_index(drop=True)

species_seen = set()
richness_curve = []

for species in df_sorted["species"]:
    species_seen.add(species)
    richness_curve.append(len(species_seen))

richness_curve = np.array(richness_curve)
records = np.arange(1, len(richness_curve) + 1)

In [ ]:
## Plot Species Accumulation Curve

plt.figure(figsize=(10,6))

plt.plot(
    records,
    richness_curve,
    linewidth=2,
    color="tab:green"
)

plt.xlabel("Number of occurrence records")
plt.ylabel("Observed species richness")

plt.title("Species Accumulation Curve (GBIF Amphibians - Brazil)")

plt.grid(alpha=0.25)

plt.tight_layout()

plt.show()

## Interpretation

The species accumulation curve shows a rapid initial increase in species richness, followed by a progressive slowdown in the rate of new species detection as sampling effort increases.

Although the curve does not reach a clear asymptote, it begins to flatten at higher sampling intensities, suggesting partial sampling saturation at the national scale.

This pattern indicates that the GBIF dataset captures a substantial portion of the known amphibian diversity in Brazil, but additional sampling—particularly in underrepresented regions—would likely continue to yield new species records.

Importantly, the non-asymptotic shape of the curve reflects both true biological diversity and uneven sampling effort across space, time, and taxonomic groups.

## Rarefaction Curve (Permutation-Based)

To reduce the effect of sample ordering in the species accumulation process, a rarefaction approach was applied using random permutations of the dataset.

In this method, occurrence records are randomly shuffled multiple times, and species accumulation is computed for each iteration. The final curve represents the average accumulation trajectory across all permutations.

This approach provides a more robust estimate of species discovery rates and reduces stochastic artifacts associated with single-sequence accumulation curves.

In [ ]:
## Rarefaction curve via random permutations
## Goal: reduce ordering bias in species accumulation

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df_valid = df[df["spatial_quality"] == "valid"].copy()

n_permutations = 50  

n_records = len(df_valid)

all_curves = np.zeros((n_permutations, n_records))

for i in range(n_permutations):

    shuffled = df_valid.sample(frac=1).reset_index(drop=True)

    seen_species = set()
    richness_curve = []

    for sp in shuffled["species"]:
        seen_species.add(sp)
        richness_curve.append(len(seen_species))

    all_curves[i, :] = richness_curve

# permutations mean
mean_curve = all_curves.mean(axis=0)

# axis x
x = np.arange(1, n_records + 1)

In [ ]:
## Rarefaction curve (mean of permutations)

plt.figure(figsize=(10,6))

# individual lines
for i in range(n_permutations):
    plt.plot(
        x,
        all_curves[i],
        color="gray",
        alpha=0.1
    )

# mean curve
plt.plot(
    x,
    mean_curve,
    color="tab:green",
    linewidth=2.5,
    label="Mean rarefaction curve"
)

std_curve = all_curves.std(axis=0)

plt.xlabel("Number of occurrence records")
plt.ylabel("Species richness")

plt.title("Rarefaction Curve (Permutation-based)")

plt.grid(alpha=0.25)

plt.fill_between(x, mean_curve-std_curve, mean_curve+std_curve, alpha=0.2)

plt.legend()

plt.tight_layout()

plt.show()

## Interpretation

The rarefaction analysis confirms the general pattern observed in the species accumulation curve, while reducing the influence of record ordering.

The averaged curve shows a rapid initial increase in species richness, followed by a gradual deceleration in the rate of new species discovery. However, the curve does not reach a clear asymptote, suggesting that the dataset remains incompletely sampled at the national scale.

The similarity between the accumulation and rarefaction curves indicates that the observed pattern is robust and not an artifact of record ordering. Together, these results suggest that while the GBIF dataset captures a substantial portion of known amphibian diversity in Brazil, additional sampling effort—particularly in underrepresented regions and taxa—would likely continue to increase observed richness.

## GLM Temporal Trend

To evaluate whether occurrence records show a significant temporal trend, a Generalized Linear Model (GLM) was fitted using year as the predictor variable.

This analysis helps distinguish between increases driven by true changes in biodiversity data collection versus artifacts of sampling effort and digitization processes over time.

In [ ]:
## GLM Temporal Trend - data aggregation

import pandas as pd
import numpy as np

import statsmodels.api as sm

df_valid = df[df["spatial_quality"] == "valid"].copy()

year_counts = (
    df_valid["year"]
    .value_counts()
    .dropna()
    .sort_index()
)

year_df = year_counts.reset_index()
year_df.columns = ["year", "records"]

year_df.head()

In [ ]:
## Temporal trend visualization

plt.figure(figsize=(10,6))

plt.plot(
    year_df["year"],
    year_df["records"],
    color="tab:blue",
    linewidth=2
)

plt.xlabel("Year")
plt.ylabel("Number of occurrence records")

plt.title("Temporal Distribution of GBIF Records")

plt.grid(alpha=0.25)

plt.tight_layout()

plt.show()

In [ ]:
## GLM (Poisson regression)

X = sm.add_constant(year_df["year"])
y = year_df["records"]

glm_poisson = sm.GLM(
    y,
    X,
    family=sm.families.Poisson()
)

result = glm_poisson.fit()

print(result.summary())

In [ ]:
## Predicted temporal trend

year_df["predicted"] = result.predict(X)

plt.figure(figsize=(10,6))

plt.scatter(
    year_df["year"],
    year_df["records"],
    alpha=0.6,
    label="Observed"
)

plt.plot(
    year_df["year"],
    year_df["predicted"],
    color="red",
    linewidth=2,
    label="GLM fit (Poisson)"
)

plt.xlabel("Year")
plt.ylabel("Number of records")

plt.title("Temporal Trend in Occurrence Records (GLM)")

plt.legend()

plt.grid(alpha=0.25)

plt.tight_layout()

plt.show()

## Interpretation

The Poisson GLM reveals a strong and highly significant positive temporal trend in the number of occurrence records over time (p < 0.001).

The estimated coefficient for year (β = 0.0301) indicates an exponential increase in the number of records through time, consistent with the log-link function of the model. This confirms that record accumulation is not linear, but accelerates rapidly in more recent decades.

This pattern reflects historical and technological processes rather than biological change. The sharp increase in records after ~1950 corresponds to the expansion of museum-based biodiversity surveys, while the post-2000 increase aligns with large-scale digitization efforts and the rise of open biodiversity platforms such as GBIF.

Therefore, the temporal structure of the dataset is primarily driven by sampling effort and data mobilization rather than ecological dynamics, and should be accounted for in any downstream biodiversity inference.

## Interactive Spatial Distribution (Plotly Maps)

To explore the spatial structure of occurrence records, an interactive map was generated using Plotly.

This visualization allows dynamic inspection of geographic patterns, including clustering of records, geographic gaps, and potential sampling hotspots across Brazil.

Spatial filtering based on data quality was applied to remove coordinate outliers before visualization.

In [ ]:
## Interactive map of valid occurrence records

import plotly.express as px

df_map = df[df["spatial_quality"] == "valid"].copy()

fig = px.scatter_geo(
    df_map,
    lat="decimallatitude",
    lon="decimallongitude",
    opacity=0.3,
    size_max=2
)

fig.update_layout(
    title="GBIF Amphibian Occurrence Records - Brazil",
    geo=dict(
        scope="south america",
        showland=True
    )
)

fig.show()

## Temporal-Spatial Distribution of Records

To investigate how spatial sampling patterns vary over time, occurrence records were colored by decade.

This allows identification of shifts in sampling coverage and expansion of biodiversity data collection across Brazilian regions.

In [ ]:
## Interactive temporal map (colored by decade)

fig = px.scatter_geo(
    df_map,
    lat="decimallatitude",
    lon="decimallongitude",
    color="decade",
    opacity=0.4,
    hover_name="species",
    hover_data=["institutioncode", "year"],
    color_continuous_scale="Viridis"
)

fig.update_layout(
    title="Temporal Distribution of Amphibian Records in Brazil",
    geo=dict(
        scope="south america",
        projection_type="natural earth"
    )
)

fig.show()

In [ ]:
## Density-style spatial visualization

fig = px.density_mapbox(
    df_map,
    lat="decimallatitude",
    lon="decimallongitude",
    radius=5,
    center=dict(lat=-14, lon=-55),
    zoom=3,
    mapbox_style="carto-positron"
)

fig.update_layout(
    title="Spatial Density of Amphibian Occurrence Records in Brazil"
)

fig.show()

## Interpretation

The spatial distribution of occurrence records reveals strong geographic clustering, with higher sampling intensity in southeastern and northern Brazil.

This pattern is consistent with the concentration of research institutions, accessibility gradients, and historical sampling priorities.

When combined with temporal information, the maps show a clear expansion of spatial coverage over time, particularly after the 2000s, reflecting the growth of digitized biodiversity databases and citizen science contributions.

Despite this expansion, large spatial gaps remain, particularly in remote regions of the Amazon and central Brazil, indicating persistent geographic sampling bias in the dataset.

## Final Conclusions

This study explored a large-scale GBIF occurrence dataset of Brazilian amphibians through a combination of data engineering, exploratory analysis, and statistical modeling.

Across spatial, temporal, institutional, and taxonomic dimensions, the dataset exhibits strong and consistent sampling biases. Occurrence records are highly concentrated in specific geographic regions, particularly in southeastern and northern Brazil, and are unevenly distributed across institutions, reflecting historical research capacity and data mobilization efforts.

Temporal analysis revealed a pronounced exponential increase in records over time, strongly associated with the expansion of museum digitization and biodiversity informatics initiatives. The Poisson GLM confirmed this trend as statistically significant, indicating that temporal variation in the dataset is primarily driven by sampling effort rather than ecological processes.

Taxonomic analyses further highlighted substantial imbalance in species representation. A small fraction of species accounts for the majority of occurrence records, with only 199 species representing approximately 80% of the dataset. This strong dominance structure reflects both ecological factors and long-standing research biases toward certain taxa.

Species accumulation and rarefaction curves indicate that, although the dataset captures a large portion of known amphibian diversity in Brazil, it remains incomplete and does not reach full sampling saturation. This suggests that additional sampling—particularly in underrepresented regions and taxa—would likely continue to reveal new records.

Overall, while the dataset provides a rich and valuable resource for biodiversity research, it is fundamentally shaped by heterogeneous sampling processes. Any ecological inference drawn from these data must explicitly account for spatial, temporal, taxonomic, and institutional biases.

Despite these limitations, the dataset remains highly informative when analyzed under a framework that acknowledges and quantifies these sources of bias.

This workflow demonstrates how raw biodiversity data can be transformed into analytically meaningful insights through structured data engineering, exploratory analysis, and bias-aware interpretation.